# Taller — Limpieza y Análisis de Datos con Pandas y NumPy
**Objetivo:** Aplicar técnicas de limpieza, transformación y análisis de datos utilizando Pandas y NumPy a partir de una base de datos con inconsistencias similares a escenarios reales.

---
Archivo requerido: `ventas_sucias_5000.csv` — debe estar en la misma carpeta que este notebook.


 Librerías e importaciones

In [20]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# pd.read_csv() lee el archivo CSV y lo convierte en un DataFrame de Pandas.
# El archivo debe estar en la misma carpeta que este notebook.
df = pd.read_csv('ventas_sucias_5000.csv')

# create_engine() crea una base de datos SQLite en memoria (no guarda en disco).
# df.to_sql() carga el DataFrame como una tabla SQL llamada 'tabla_usuarios'.
engine = create_engine('sqlite:///:memory:')
df.to_sql('tabla_usuarios', con=engine, if_exists='replace', index=False)

print("Archivo cargado correctamente")

Archivo cargado correctamente


---
## Parte 1 — Verificación de Datos
**Objetivo:** Conocer la estructura del dataset antes de limpiarlo.

### 1.1 Primeras filas · `df.head()`

In [21]:
# head() muestra las primeras 5 filas para una vista rápida del contenido.
# En Jupyter no necesita print() — el DataFrame se renderiza como tabla automáticamente.
df.head()

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,1326.0,NaN,peru,Efectivo,2024-01-01 00:00:00
1,Luisa,Laptop,55.0,2,chile,Tarjeta,2024-01-01 01:00:00
2,Carlos,Monitor,1203.0,9,Colombia,Efectivo,2024-01-01 02:00:00
3,Luisa,Monitor,1304.0,3,Perú,TRANSFERENCIA,2024-01-01 03:00:00
4,Luisa,Monitor,426.0,6,chile,Tarjeta,2024-01-01 04:00:00


### 1.2 Información general · `df.info()`

In [22]:
# info() muestra el tipo de dato de cada columna y cuántos valores no nulos hay.
# Es útil para detectar columnas con tipos incorrectos (ej: números guardados como texto).
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      5000 non-null   object 
 1   producto     5000 non-null   object 
 2   precio       4950 non-null   float64
 3   cantidad     4950 non-null   object 
 4   pais         5000 non-null   object 
 5   metodo_pago  5000 non-null   object 
 6   fecha        5000 non-null   object 
dtypes: float64(1), object(6)
memory usage: 273.6+ KB


### 1.3 Resumen estadístico · `df.describe()`

In [23]:
# describe() genera estadísticas descriptivas: count, mean, std, min, 25%, 50%, 75%, max.
df.describe()

,precio
count,4950.000000
mean,5053.823434
std,63379.982009
min,10.000000
25%,531.000000
50%,1027.000000
75%,1519.000000
max,999999.000000


### 1.4 Filas, columnas, nulos y duplicados

In [24]:
# shape devuelve (filas, columnas).
print("Filas y columnas:", df.shape)

# isnull().sum() cuenta los valores NaN por columna.
print("\nValores faltantes (NaN):")
print(df.isnull().sum())

# Filtramos precio == 999999 — valor centinela de sistema (indica dato faltante o error).
resultados_999999 = df[df['precio'] == 999999]
print(f"\nFilas con precio = 999999: {len(resultados_999999)}")
print(resultados_999999)

# duplicated().sum() cuenta filas idénticas.
print(f"\nValores duplicados: {df.duplicated().sum()}")

Filas y columnas: (5000, 7)

Valores faltantes (NaN):
cliente         0
producto        0
precio         50
cantidad       50
pais            0
metodo_pago     0
fecha           0
dtype: int64

Filas con precio = 999999: 20
     cliente producto    precio cantidad      pais    metodo_pago  \
771    Luisa  Teclado  999999.0        1       col        Tarjeta   
1123     Ana   Laptop  999999.0        2      peru       Efectivo   
1159   Pedro  Teclado  999999.0        2      Perú  TRANSFERENCIA   
1165   Luisa    Mouse  999999.0        3     Chile        Tarjeta   
1188   Pedro    Mouse  999999.0        4      Perú  transferencia   
1195     Ana   Laptop  999999.0        8  Colombia        Tarjeta   
1225     Ana  Monitor  999999.0        5       COL  transferencia   
1242    Juan    Mouse  999999.0        5  Colombia  transferencia   
1277   Pedro  Celular  999999.0        3     Chile        Tarjeta   
1311  Carlos  Teclado  999999.0        4       COL  TRANSFERENCIA   
1885   Maria    M

 Respuestas Parte 1

¿Cuántas filas y columnas tiene la base de datos?
 El dataset tiene 5 000 filas y 7 columnas.

¿Qué tipos de datos se identificaron? 
> - `precio`: float64
> - `cantidad`: object *(texto — debería ser int)*
> - `pais`, `metodo_pago`, `fecha`: object *(texto)*

¿Encuentras columnas con problemas o inconsistencias?
> - `cantidad` contiene el texto `"three"` en lugar del número 3.
> - `precio` tiene 20 registros con valor `999999` (outlier centinela).
> - `pais` y `metodo_pago` tienen variantes en mayúsculas/minúsculas.
> - `fecha` está guardada como texto con hora incluida.

 Parte 2 — Limpieza de Datos
Objetivo: Corregir tipos, estandarizar texto, manejar nulos y outliers.

### 2.1 Eliminar duplicados

In [25]:
# drop_duplicates() elimina filas idénticas.
# .copy() evita el warning "SettingWithCopyWarning" al modificar columnas.
df1 = df.drop_duplicates().copy()
print(f"Filas después de eliminar duplicados: {len(df1)}")

Filas después de eliminar duplicados: 5000


### 2.2 Estandarización de `pais`

In [26]:
# str.lower() convierte a minúsculas → "Colombia", "COLOMBIA" → "colombia"
# str.strip() elimina espacios al inicio y al final.
# replace(" ", np.nan) → espacios sueltos se tratan como vacíos.
# fillna("desconocido") rellena los NaN con un valor por defecto.
for col in ["pais"]:
    df1[col] = df1[col].str.lower().str.strip()
    df1[col] = df1[col].replace(" ", np.nan)
    df1[col] = df1[col].fillna("desconocido")

# Corrección de variantes mal escritas.
df1["pais"] = df1["pais"].replace({
    "col":       "colombia",
    "colomobia": "colombia",
    "perú":      "peru"
})

print("Valores únicos en pais:", sorted(df1["pais"].unique()))

Valores únicos en pais: ['chile', 'colombia', 'peru']


### 2.3 Corrección de `cantidad`

In [27]:
# astype(str) convierte la columna a texto para poder usar .replace().
# .replace({"three": "3"}) cambia el texto "three" por el string "3".
# pd.to_numeric(errors='coerce') convierte a número; lo que no pueda → NaN.
df1['cantidad'] = df1['cantidad'].astype(str).replace({"three": "3"})
df1['cantidad'] = pd.to_numeric(df1['cantidad'], errors='coerce')
print(f"Nulos en cantidad tras conversión: {df1['cantidad'].isna().sum()}")

Nulos en cantidad tras conversión: 50


### 2.4 Estandarización de `metodo_pago`

In [28]:
# str.lower() unifica "TRANSFERENCIA", "Transferencia", "transferencia" → "transferencia"
df1['metodo_pago'] = df1['metodo_pago'].str.lower()
print("Valores únicos en metodo_pago:", sorted(df1["metodo_pago"].unique()))

Valores únicos en metodo_pago: ['efectivo', 'tarjeta', 'transferencia']


### 2.5 Manejo del outlier en `precio`

In [29]:
# Reemplazamos 999999 por NaN — es un valor centinela que indica dato faltante.
df1['precio'] = df1['precio'].replace(999999, np.nan)
print(f"Nulos en precio tras reemplazo de 999999: {df1['precio'].isna().sum()}")

Nulos en precio tras reemplazo de 999999: 70


### 2.6 Corrección de fechas

In [30]:
# pd.to_datetime() convierte "2024-01-01 00:00:00" al tipo datetime.
# errors='coerce' convierte fechas inválidas en NaT (NaN para fechas).
# fillna reemplaza fechas nulas con la fecha centinela "1900-01-01".
df1['fecha'] = pd.to_datetime(df1['fecha'], errors='coerce')
df1['fecha'] = df1['fecha'].fillna(pd.to_datetime('1900-01-01'))
print("Tipo de fecha:", df1['fecha'].dtype)

Tipo de fecha: datetime64[ns]


### 2.7 Imputación de nulos con la mediana

In [31]:
# Se usa la mediana en lugar del promedio porque es robusta ante outliers.
# El promedio se distorsiona con valores extremos; la mediana no.
df1["cantidad"] = df1["cantidad"].fillna(df1["cantidad"].median())
df1["precio"]   = df1["precio"].fillna(df1["precio"].median())

print("Limpieza completada")
print(f"Forma del dataset limpio: {df1.shape}")
print(f"\nValores nulos restantes:\n{df1.isnull().sum()}")

Limpieza completada
Forma del dataset limpio: (5000, 7)

Valores nulos restantes:
cliente        0
producto       0
precio         0
cantidad       0
pais           0
metodo_pago    0
fecha          0
dtype: int64


### Vista del dataset limpio

In [32]:
df1.head()

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha
0,Maria,Monitor,1326.0,5.0,peru,efectivo,2024-01-01 00:00:00
1,Luisa,Laptop,55.0,2.0,chile,tarjeta,2024-01-01 01:00:00
2,Carlos,Monitor,1203.0,9.0,colombia,efectivo,2024-01-01 02:00:00
3,Luisa,Monitor,1304.0,3.0,peru,transferencia,2024-01-01 03:00:00
4,Luisa,Monitor,426.0,6.0,chile,tarjeta,2024-01-01 04:00:00


### Respuestas Parte 2

**¿Qué problemas encontraste?**
> - 50 valores nulos en `precio` y `cantidad`.
> - 20 filas con `precio = 999999` (valor centinela / error de sistema).
> - `cantidad` tenía el texto `"three"` en lugar del entero 3.
> - `pais` tenía 7 variantes del mismo valor (`col`, `COL`, `Colombia`, etc.).
> - `metodo_pago` tenía mayúsculas inconsistentes (`TRANSFERENCIA` vs `transferencia`).

**¿Cómo los solucionaste?**
> - Texto: `str.lower()` + `str.strip()` + `replace()` para estandarizar.
> - Outlier 999999: reemplazado por NaN con `.replace()`.
> - `"three"`: convertido a `"3"` con `.replace()` antes de `to_numeric()`.
> - Nulos: imputados con la mediana para no distorsionar los promedios.

**¿Eliminaste registros? ¿Por qué?**
> Solo se eliminaron duplicados (0 encontrados en este dataset). No se eliminaron filas por nulos — se imputaron con la mediana para conservar la mayor cantidad posible de registros.

---
## Parte 3 — Análisis con Pandas
**Objetivo:** Calcular métricas de ventas y detectar patrones por agrupación.

### 3.1 Columna `total` y métricas principales

In [33]:
# Creamos la columna 'total' multiplicando precio por cantidad.
# Pandas aplica la operación fila por fila de forma vectorizada (sin bucles).
df1["total"] = df1["cantidad"] * df1["precio"]

total_vendido_pd   = df1["total"].sum()    # suma de todos los totales
promedio_ventas_pd = df1["total"].mean()   # promedio aritmético
venta_maxima_pd    = df1["total"].max()    # valor más alto
venta_minima_pd    = df1["total"].min()    # valor más bajo

print(f"Total vendido:     ${total_vendido_pd:>15,.2f}")
print(f"Promedio ventas:   ${promedio_ventas_pd:>15,.2f}")
print(f"Venta máxima:      ${venta_maxima_pd:>15,.2f}")
print(f"Venta mínima:      ${venta_minima_pd:>15,.2f}")

Total vendido:     $  25,119,629.00
Promedio ventas:   $       5,023.93
Venta máxima:      $      17,955.00
Venta mínima:      $          10.00


### 3.2 Top 5 productos con mayor valor vendido

In [34]:
# groupby('producto') agrupa todas las filas por nombre de producto.
# ['total'].sum() suma los totales de cada grupo.
# nlargest(5) devuelve los 5 productos con mayor suma.
top5_productos = df1.groupby('producto')['total'].sum().nlargest(5).reset_index()
top5_productos.columns = ['Producto', 'Total Vendido']
top5_productos['Total Vendido'] = top5_productos['Total Vendido'].apply(lambda x: f"${x:,.2f}")
top5_productos

,Producto,Total Vendido
0,Mouse,"$5,414,526.00"
1,Laptop,"$5,109,312.00"
2,Monitor,"$4,948,044.00"
3,Celular,"$4,882,707.00"
4,Teclado,"$4,765,040.00"


### 3.3 País con más ventas

In [35]:
# groupby('pais')['total'].sum() suma las ventas por país.
# idxmax() devuelve el nombre del país con la suma más alta.
pais_top = df1.groupby('pais')['total'].sum().sort_values(ascending=False).reset_index()
pais_top.columns = ['País', 'Total Vendido']
pais_top['Total Vendido'] = pais_top['Total Vendido'].apply(lambda x: f"${x:,.2f}")
pais_top

,País,Total Vendido
0,colombia,"$10,886,212.00"
1,chile,"$7,225,747.00"
2,peru,"$7,007,670.00"


---
## Parte 4 — Introducción a NumPy
**Objetivo:** Trabajar con arrays de NumPy para cálculo eficiente.

In [36]:
# to_numpy() convierte el DataFrame (dos columnas) en un array 2D de NumPy.
# Un array 2D tiene forma (filas, columnas) — en este caso (5000, 2).
data = df1[['precio', 'cantidad']].to_numpy()
print(f"Array creado → shape: {data.shape} | dtype: {data.dtype}")

# data[:, 0] = todas las filas (:), columna índice 0 → precios
# data[:, 1] = todas las filas (:), columna índice 1 → cantidades
precios    = data[:, 0]
cantidades = data[:, 1]

# Vectorización: NumPy multiplica los dos arrays elemento a elemento
# sin necesidad de un bucle for. Entre 10x y 100x más rápido que un for.
totales = precios * cantidades
print(f"Primeros 5 totales calculados: {totales[:5].round(2)}")
print("Conversión y cálculo vectorizado con NumPy completado.")

Array creado → shape: (5000, 2) | dtype: float64
Primeros 5 totales calculados: [ 6630.   110. 10827.  3912.  2556.]
Conversión y cálculo vectorizado con NumPy completado.


---
## Parte 5 — Análisis con NumPy
**Objetivo:** Calcular métricas usando funciones nativas de NumPy.

In [37]:
# np.sum() suma todos los elementos del array totales.
suma_total_np = np.sum(totales)

# np.mean() calcula el promedio aritmético del array.
promedio_ventas_np = np.mean(totales)

# np.max() devuelve el valor más alto del array.
venta_maxima_np = np.max(totales)

# totales > 1000 genera un array booleano (True/False).
# np.sum() cuenta los True (Python trata True=1, False=0).
ventas_mayores_1000 = np.sum(totales > 1000)

print(f"Suma total (NumPy):         ${suma_total_np:>15,.2f}")
print(f"Promedio de ventas (NumPy): ${promedio_ventas_np:>15,.2f}")
print(f"Venta máxima (NumPy):       ${venta_maxima_np:>15,.2f}")
print(f"Cantidad de ventas > $1000: {ventas_mayores_1000} ({ventas_mayores_1000/len(totales)*100:.1f}%)")

Suma total (NumPy):         $  25,119,629.00
Promedio de ventas (NumPy): $       5,023.93
Venta máxima (NumPy):       $      17,955.00
Cantidad de ventas > $1000: 4248 (85.0%)


### Respuestas Parte 5

**¿Qué ventajas observas al usar NumPy?**
> NumPy opera sobre arrays completos en lenguaje C, sin bucles en Python. Es entre 10x y 100x más rápido que iterar con `for`, consume menos memoria que un DataFrame de Pandas, y sus funciones (`sum`, `mean`, `max`) están altamente optimizadas para cálculos numéricos a gran escala.

**¿Qué significa "vectorización"?**
> Es aplicar una operación matemática a todos los elementos de un array de una sola vez, sin escribir un bucle explícito. Por ejemplo, `totales = precios * cantidades` multiplica cada precio por su cantidad correspondiente en una sola instrucción, en lugar de hacer un `for` elemento por elemento. NumPy ejecuta esa operación internamente de forma masiva y eficiente.

**¿Qué hace la expresión `data[:, 0]`?**
> Selecciona **todas las filas** (`:`) de la columna con **índice 0** del array 2D. El array `data` tiene forma `(5000, 2)`: 5000 filas y 2 columnas.
> - `data[:, 0]` → columna 0 → **precios** (array de 5000 valores)
> - `data[:, 1]` → columna 1 → **cantidades** (array de 5000 valores)

---
## Parte 6 — Interpretación de Resultados
**Objetivo:** Analizar los resultados y tomar decisiones basadas en los datos.

In [38]:
mediana_total = np.median(totales)
diferencia_pct = abs(promedio_ventas_np - mediana_total) / mediana_total * 100

print(f"Promedio : ${promedio_ventas_np:,.2f}")
print(f"Mediana  : ${mediana_total:,.2f}")
print(f"Diferencia: {diferencia_pct:.1f}%")

Promedio : $5,023.93
Mediana  : $3,914.50
Diferencia: 28.3%


### Respuestas Parte 6

**1. ¿Los resultados obtenidos tienen sentido?**
> Sí. Después de la limpieza profunda (corrección de outliers `999999`, imputación con mediana, estandarización de texto y corrección del tipo de dato en `cantidad`) los datos reflejan un comportamiento coherente: ventas distribuidas entre los 3 países y 5 productos de forma homogénea, sin valores extremos que distorsionen el análisis.

**2. ¿Detectaste valores sospechosos?**
> Sí. Se detectaron tres tipos:
> - `precio = 999999`: valor centinela de sistema (20 filas).
> - `cantidad = "three"`: texto en columna numérica.
> - `pais` con variantes: `"col"`, `"COL"`, `"colombia"` representaban lo mismo.
>
> Todos fueron corregidos antes del análisis.

**3. ¿El promedio representa correctamente los datos?**
> No del todo. El promedio puede verse afectado por valores extremos (outliers). La mediana es más representativa de la venta típica porque no se distorsiona con valores atípicos. La diferencia entre ambos en este dataset supera el 28%, lo que indica sesgo. Para reportes ejecutivos se recomienda reportar la **mediana**.

**4. ¿Qué decisiones tomarías si esta fuera información real de negocio?**
> - Limitar el sistema de recolección para rechazar valores inválidos (`999999`, texto en campos numéricos) en el momento del ingreso.
> - Usar un catálogo controlado (dropdown) para `pais` y `metodo_pago`, eliminando variabilidad por escritura libre.
> - Usar la **mediana** como KPI principal de venta típica en dashboards.
> - Investigar si las 20 filas con `precio=999999` corresponden a ventas reales no registradas o a errores del sistema.
> - Revisar por qué `cantidad` admitía texto: validar el formulario o API de entrada.